# Colab Linear-Probe Training for Experiment 1

This notebook trains Experiment 1 frozen-feature linear probes on a Colab GPU using a zipped copy of the repo (and optionally a separate data archive) stored in Google Drive.

It is designed for the case where your local machine is too slow for the probe grid. Feature extraction has already produced `.npz` caches; this notebook only runs the cheap-but-numerous probe training, results aggregation, and figure generation stages on GPU.

Before running:

1. In Colab, choose **Runtime -> Change runtime type -> GPU** (T4, L4, or A100 all work; TPU is not required - the probes are tiny).
2. Upload this notebook to Colab.
3. Edit the config cell below to point at your zipped repo (and optional data archive) in Google Drive.
4. Run all cells top to bottom.

In [ ]:
from pathlib import Path

# Required: path inside Drive to the zipped repo (the .zip you uploaded).
# Examples:
#   /content/drive/MyDrive/cv-project.zip
#   /content/drive/MyDrive/cv-project-2026-05-12.zip
REPO_ZIP_IN_DRIVE = "/content/drive/MyDrive/cv-project.zip"

# Optional: path inside Drive to a separate compressed `data/exp1_bounded` archive
# (.zip or .tar.gz). Leave empty when the repo zip already contains the bounded
# data tree under `data/exp1_bounded/`.
DRIVE_DATA_ARCHIVE = ""

# Hydra config inside the repo. The bounded run config is what produced the
# data/exp1_bounded/* outputs on the local machine.
CONFIG_PATH_IN_REPO = "configs/exp1_bounded.yaml"

# Where outputs/exp1_bounded should be archived back to in Drive.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/exp1_bounded_outputs"

# Local Colab workspace (fast SSD, ephemeral).
WORK_DIR = Path("/content/exp1_colab")
REPO_DIR = WORK_DIR / "cv-project"
REPO_EXTRACT_DIR = WORK_DIR / "repo_extract"
DATA_EXTRACT_DIR = WORK_DIR / "data_extract"

# Probe training overrides. Leave as None to use values from the Hydra config.
PROBE_EPOCHS = None       # e.g. 30 for a longer run.
PROBE_BATCH_SIZE = 512    # GPU is fast; bigger batches finish probes faster.
PROBE_LR = None
PROBE_WEIGHT_DECAY = None
PROBE_SEED = None

# Optional filters. Leave empty to use everything from the config.
MODELS = []   # e.g. ["clip_vit_b16", "clip_vit_l14", "dinov2_vit_b"]
LAYERS = []   # e.g. ["final", "layer4", "layer8", "layer12"]
TASKS = []    # e.g. ["surface_normal_aggregate", "relative_depth_regions"]

FORCE_REPO_REFRESH = False  # Re-extract the repo zip even if a copy is present.
FORCE_DATA_REFRESH = False  # Re-extract the data archive even if a copy is present.
RUN_FIGURES = True          # Set False to skip qualitative tables / matplotlib plots.
EXPORT_OUTPUTS_TO_DRIVE = True

print("Workspace:", WORK_DIR)

## Mount Drive and Check GPU

Linear probes are small (one Linear layer on cached features), but GPU still wins because there are roughly `models * layers * tasks * (within_texture + cross_texture)` probes to train.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import subprocess
import sys
import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    subprocess.run(["nvidia-smi"], check=False)
else:
    raise RuntimeError(
        "No CUDA GPU is available. In Colab, switch Runtime type to GPU before running this notebook."
    )

## Install Minimal Probe Dependencies

We install only what the probe / aggregate / figure stages need, not Blender, pyrender, transformers, or open_clip. That keeps Colab setup fast.

In [ ]:
packages = [
    "hydra-core>=1.3.2",
    "omegaconf>=2.3.0",
    "pandas>=2.0.0",
    "pyarrow>=14.0.0",
    "Pillow>=10.0.0",
    "tqdm>=4.66.0",
    "scikit-learn>=1.3.0",
    "matplotlib>=3.8.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Probe-stage dependencies installed.")

## Extract Repo from Drive Zip

The repo zip is extracted to local SSD, not into Drive. Drive is much slower for many small files.

In [ ]:
import shutil
import tarfile
import zipfile

def run(cmd, *, cwd=None, env=None, check=True):
    printable = " ".join(str(x) for x in cmd)
    print("+", printable)
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {printable}")
    return result

def safe_extract_archive(archive_path: Path, dest: Path) -> None:
    archive_path = Path(archive_path)
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    resolved_dest = dest.resolve()
    print(f"Extracting {archive_path} -> {dest}")
    suffix = archive_path.suffix.lower()
    if suffix == ".zip":
        with zipfile.ZipFile(archive_path) as zf:
            for member in zf.infolist():
                target = (dest / member.filename).resolve()
                if not str(target).startswith(str(resolved_dest)):
                    raise RuntimeError(f"Unsafe archive member: {member.filename}")
            zf.extractall(dest)
        return
    if tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as tf:
            for member in tf.getmembers():
                target = (dest / member.name).resolve()
                if not str(target).startswith(str(resolved_dest)):
                    raise RuntimeError(f"Unsafe archive member: {member.name}")
            tf.extractall(dest)
        return
    raise ValueError(f"Unsupported archive format: {archive_path}")

def find_repo_root(root: Path) -> Path:
    hits = sorted(root.rglob("scripts/train_all_exp1_probes.py"))
    if not hits:
        raise FileNotFoundError("Could not find scripts/train_all_exp1_probes.py in the extracted repo")
    return hits[0].parents[1]

WORK_DIR.mkdir(parents=True, exist_ok=True)

if FORCE_REPO_REFRESH:
    for path in (REPO_DIR, REPO_EXTRACT_DIR):
        if path.exists():
            print("Removing", path)
            shutil.rmtree(path)

if (REPO_DIR / "scripts" / "train_all_exp1_probes.py").is_file():
    print("Repo already prepared:", REPO_DIR)
else:
    repo_zip = Path(REPO_ZIP_IN_DRIVE)
    if not repo_zip.is_file():
        raise FileNotFoundError(f"Missing REPO_ZIP_IN_DRIVE: {repo_zip}")
    safe_extract_archive(repo_zip, REPO_EXTRACT_DIR)
    repo_root = find_repo_root(REPO_EXTRACT_DIR)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.move(str(repo_root), str(REPO_DIR))

sys.path.insert(0, str(REPO_DIR))
os.environ["CV_PROJECT_ROOT"] = str(REPO_DIR)
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ.setdefault("HF_HOME", str(WORK_DIR / "hf_cache"))

print("Repo ready:", REPO_DIR)
print("CV_PROJECT_ROOT:", os.environ["CV_PROJECT_ROOT"])

## Provide Bounded Run Data

The probe stage needs three things under `REPO_DIR/data/exp1_bounded`:

- `features/<model>/<layer>.npz` for every (model, layer) in the config.
- `labels/labels_<task>.parquet` for every enabled task.
- `manifests/render_valid.parquet` (used to join features, labels, and splits).

If `DRIVE_DATA_ARCHIVE` is set, this cell extracts it and symlinks its `exp1_bounded` directory into the repo. Otherwise it relies on the repo zip already shipping those files.

In [ ]:
def find_exp1_bounded_root(root: Path) -> Path:
    if root.name == "exp1_bounded" and (root / "manifests").exists():
        return root
    candidates = [
        p for p in root.rglob("exp1_bounded")
        if p.is_dir() and (p / "manifests").exists()
    ]
    if not candidates:
        raise FileNotFoundError("Could not find an extracted exp1_bounded directory in the data archive")
    return sorted(candidates, key=lambda p: len(str(p)))[0]

repo_data_path = REPO_DIR / "data" / "exp1_bounded"
repo_data_path.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_DATA_ARCHIVE.strip():
    if FORCE_DATA_REFRESH and DATA_EXTRACT_DIR.exists():
        print("Removing existing data extraction dir:", DATA_EXTRACT_DIR)
        shutil.rmtree(DATA_EXTRACT_DIR)
    archive_path = Path(DRIVE_DATA_ARCHIVE)
    if not archive_path.is_file():
        raise FileNotFoundError(f"Missing DRIVE_DATA_ARCHIVE: {archive_path}")
    if not DATA_EXTRACT_DIR.exists():
        safe_extract_archive(archive_path, DATA_EXTRACT_DIR)
    data_root = find_exp1_bounded_root(DATA_EXTRACT_DIR)
    if repo_data_path.exists() or repo_data_path.is_symlink():
        if repo_data_path.is_symlink():
            repo_data_path.unlink()
        else:
            shutil.rmtree(repo_data_path)
    os.symlink(data_root, repo_data_path, target_is_directory=True)
    print("Linked bounded data:", repo_data_path, "->", data_root)
else:
    if not repo_data_path.exists():
        raise FileNotFoundError(
            f"{repo_data_path} does not exist. Either include data/exp1_bounded in the "
            "repo zip, or set DRIVE_DATA_ARCHIVE in the config cell."
        )
    print("Using bounded data shipped in repo zip:", repo_data_path)

## Verify Inputs Match the Config

Loads `configs/exp1_bounded.yaml`, then checks that every enabled `<model>/<layer>.npz` feature cache, every enabled task label parquet, and the valid render manifest exist on local disk.

In [ ]:
from exp1.config import load_exp1_config, resolve_path

config_path = REPO_DIR / CONFIG_PATH_IN_REPO
if not config_path.is_file():
    raise FileNotFoundError(f"Hydra config not found at {config_path}")
cfg = load_exp1_config(config_path)
project_root = Path(str(cfg.paths.project_root)).expanduser().resolve()
if project_root != REPO_DIR.resolve():
    raise RuntimeError(
        f"Config resolves project_root to {project_root}, expected {REPO_DIR}. "
        "Did CV_PROJECT_ROOT get unset?"
    )

feature_dir = resolve_path(project_root, str(cfg.paths.feature_dir))
labels_dir = resolve_path(project_root, str(cfg.paths.labels_dir))
manifest_path = resolve_path(project_root, str(cfg.paths.valid_render_manifest))

enabled_models = MODELS or [str(m) for m in cfg.models.enabled]
enabled_layers = LAYERS or [str(l) for l in cfg.models.layers]
enabled_tasks = TASKS or [str(t) for t in cfg.tasks.enabled]

missing = []
for model in enabled_models:
    for layer in enabled_layers:
        path = feature_dir / model / f"{layer}.npz"
        if not path.is_file():
            missing.append(str(path))
for task in enabled_tasks:
    task_label = cfg.tasks.definitions[task].get("label_path")
    if task_label is None:
        continue
    label_resolved = resolve_path(project_root, str(task_label))
    if not label_resolved.is_file():
        missing.append(str(label_resolved))
if not manifest_path.is_file():
    missing.append(str(manifest_path))

if missing:
    print("Missing inputs:")
    for path in missing:
        print(" -", path)
    raise FileNotFoundError(
        "Some probe inputs were not found. Make sure feature caches, labels, and the "
        "render_valid manifest are bundled in your repo zip or DRIVE_DATA_ARCHIVE."
    )

print("Feature dir:   ", feature_dir)
print("Labels dir:    ", labels_dir)
print("Render manifest:", manifest_path)
print("Models:", enabled_models)
print("Layers:", enabled_layers)
print("Tasks: ", enabled_tasks)

## Train Linear Probes on GPU

Runs `scripts/train_all_exp1_probes.py` with `--device cuda`. By default this evaluates within-texture and cross-texture jobs as configured in `configs/exp1_bounded.yaml`.

In [ ]:
probe_cmd = [
    sys.executable,
    str(REPO_DIR / "scripts" / "train_all_exp1_probes.py"),
    "--config", str(config_path),
    "--device", "cuda",
]
if MODELS:
    probe_cmd += ["--models", *MODELS]
if LAYERS:
    probe_cmd += ["--layers", *LAYERS]
if TASKS:
    probe_cmd += ["--tasks", *TASKS]
if PROBE_EPOCHS is not None:
    probe_cmd += ["--epochs", str(int(PROBE_EPOCHS))]
if PROBE_BATCH_SIZE is not None:
    probe_cmd += ["--batch-size", str(int(PROBE_BATCH_SIZE))]
if PROBE_LR is not None:
    probe_cmd += ["--lr", str(float(PROBE_LR))]
if PROBE_WEIGHT_DECAY is not None:
    probe_cmd += ["--weight-decay", str(float(PROBE_WEIGHT_DECAY))]
if PROBE_SEED is not None:
    probe_cmd += ["--seed", str(int(PROBE_SEED))]

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + env.get("PYTHONPATH", "")
env["CV_PROJECT_ROOT"] = str(REPO_DIR)

run(probe_cmd, cwd=REPO_DIR, env=env)

## Aggregate Probe Metrics

Combines per-probe `metrics.json` files into long results tables (`exp1_results_long.csv`, `exp1_texture_drops.csv`, `exp1_bootstrap_ci.csv`).

In [ ]:
run(
    [
        sys.executable,
        str(REPO_DIR / "scripts" / "aggregate_exp1_results.py"),
        "--config", str(config_path),
    ],
    cwd=REPO_DIR,
    env=env,
)

## Generate Figures (Quantitative and Qualitative)

Creates layer-wise metric plots and texture-drop bars, then the qualitative probe tables (input image, ground truth, per-model probe outputs) for each tested task.

The qualitative tables need access to rendered RGB images. If your repo zip / data archive does not include `data/exp1_bounded/renders/`, the qualitative tables silently skip and only the quantitative plots are written.

In [ ]:
if RUN_FIGURES:
    run(
        [
            sys.executable,
            str(REPO_DIR / "scripts" / "make_exp1_figures.py"),
            "--config", str(config_path),
        ],
        cwd=REPO_DIR,
        env=env,
    )
else:
    print("Skipping figures (RUN_FIGURES=False).")

## Save Outputs Back to Drive

Zips `outputs/exp1_bounded` (probes + results + figures) and copies the archive to Drive. Probe checkpoints can be large; results and figures alone are usually small. Adjust `INCLUDE_PROBE_CHECKPOINTS` to control archive size.

In [ ]:
INCLUDE_PROBE_CHECKPOINTS = False

if not EXPORT_OUTPUTS_TO_DRIVE:
    print("Skipping export (EXPORT_OUTPUTS_TO_DRIVE=False).")
else:
    outputs_dir = REPO_DIR / "outputs" / "exp1_bounded"
    if not outputs_dir.is_dir():
        raise FileNotFoundError(f"Expected outputs directory at {outputs_dir}")
    drive_out = Path(DRIVE_OUTPUT_DIR)
    drive_out.mkdir(parents=True, exist_ok=True)
    archive_stem = drive_out / "exp1_bounded_outputs"
    archive_path = archive_stem.with_suffix(".zip")
    if archive_path.exists():
        print("Removing previous archive:", archive_path)
        archive_path.unlink()
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for path in sorted(outputs_dir.rglob("*")):
            if not path.is_file():
                continue
            if not INCLUDE_PROBE_CHECKPOINTS and path.name == "checkpoint.pt":
                continue
            arcname = path.relative_to(outputs_dir.parent)
            zf.write(path, arcname)
    size_mb = archive_path.stat().st_size / (1024 * 1024)
    print(f"Wrote archive: {archive_path} ({size_mb:.1f} MB)")
    print("You can now download this from Drive at:", DRIVE_OUTPUT_DIR)

## Notes

- Probes are deterministic given `PROBE_SEED` (defaults to the config's `experiment.seed`).
- The repo configures `models.enabled` and `models.layers` in `configs/exp1_bounded.yaml`; setting the `MODELS` / `LAYERS` lists at the top of this notebook is a subset filter, not a way to add models or layers without matching feature caches.
- If feature caches were produced with `features.normalize: true`, do not change that flag when re-running probes; the linear head was learned against normalized features.
- TPU is not used here because each probe is a single linear layer and the bottleneck is many small jobs, not one big matmul. GPU is already faster than CPU and avoids `torch_xla` install overhead.